# Validate ODCS Data Contract Template

Validates a YAML template against the local ODCS v3.1.0 JSON schema. This notebook is intentionally template-focused: it checks structure and schema compatibility, not data quality rule execution.

In [ ]:
try:
    dbutils.widgets.text("template_path", "")
    dbutils.widgets.text("validation_output_path", "dc_nb/outputs/validation/template_validation_checks.csv")
    dbutils.widgets.text("odcs_schema_path", "dc_nb/inputs/odcs-v3.1.0.schema_ODCS.json")
    dbutils.widgets.dropdown("fail_on_error", "true", ["true", "false"])
except NameError:
    pass


def widget_value(name: str, default: str = "") -> str:
    try:
        value = dbutils.widgets.get(name)
        return value if value is not None else default
    except Exception:
        return default


TEMPLATE_PATH = widget_value("template_path")
VALIDATION_OUTPUT_PATH = widget_value("validation_output_path", "dc_nb/outputs/validation/template_validation_checks.csv")
ODCS_SCHEMA_PATH = widget_value("odcs_schema_path", "dc_nb/inputs/odcs-v3.1.0.schema_ODCS.json")
FAIL_ON_ERROR = widget_value("fail_on_error", "true").lower() == "true"

if not TEMPLATE_PATH:
    raise ValueError("template_path widget is required.")
if not VALIDATION_OUTPUT_PATH:
    raise ValueError("validation_output_path widget is required.")
if not ODCS_SCHEMA_PATH:
    raise ValueError("odcs_schema_path widget is required.")

In [ ]:
import csv
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import yaml
from jsonschema import Draft201909Validator, FormatChecker


VALIDATION_TS = datetime.now(timezone.utc).isoformat()
CHECKS: list[dict[str, Any]] = []
TEMPLATE: dict[str, Any] | None = None


def add_check(section: str, check_name: str, status: str, severity: str = "error", detail: str = "") -> None:
    CHECKS.append(
        {
            "validation_timestamp": VALIDATION_TS,
            "template_path": TEMPLATE_PATH,
            "odcs_schema_path": ODCS_SCHEMA_PATH,
            "section": section,
            "check_name": check_name,
            "status": status,
            "severity": severity,
            "detail": detail,
        }
    )


def load_template(path: str) -> dict[str, Any] | None:
    try:
        with open(path, "r", encoding="utf-8") as handle:
            parsed = yaml.safe_load(handle)
        if not isinstance(parsed, dict):
            add_check("syntax", "YAML parses to mapping", "fail", detail="Template YAML must parse to a mapping/object.")
            return None
        add_check("syntax", "YAML parses to mapping", "pass")
        return parsed
    except Exception as exc:
        add_check("syntax", "YAML parses to mapping", "fail", detail=str(exc))
        return None


def load_schema(path: str) -> dict[str, Any] | None:
    try:
        with open(path, "r", encoding="utf-8") as handle:
            schema = json.load(handle)
        add_check("schema", "ODCS JSON schema file is readable", "pass")
        return schema
    except Exception as exc:
        add_check("schema", "ODCS JSON schema file is readable", "fail", detail=str(exc))
        return None


def validate_against_schema(template: dict[str, Any], schema: dict[str, Any]) -> None:
    validator = Draft201909Validator(schema, format_checker=FormatChecker())
    errors = sorted(validator.iter_errors(template), key=lambda err: list(err.absolute_path))
    if not errors:
        add_check("odcs_schema", "Template validates against ODCS schema", "pass")
        return
    add_check("odcs_schema", "Template validates against ODCS schema", "fail", detail=f"{len(errors)} schema error(s)")
    for err in errors[:50]:
        field = "/".join(str(part) for part in err.absolute_path) or "<root>"
        add_check("odcs_schema_detail", f"Schema error at {field}", "fail", detail=err.message)
    if len(errors) > 50:
        add_check("odcs_schema_detail", "Additional schema errors truncated", "fail", detail=f"{len(errors) - 50} more error(s)")


def validate_template_conventions(template: dict[str, Any]) -> None:
    add_check("metadata", "kind is DataContract", "pass" if template.get("kind") == "DataContract" else "fail", detail=str(template.get("kind")))
    add_check("metadata", "apiVersion is v3.1.0", "pass" if template.get("apiVersion") == "v3.1.0" else "fail", detail=str(template.get("apiVersion")))
    add_check("metadata", "id is present", "pass" if bool(template.get("id")) else "fail")
    add_check("metadata", "version is present", "pass" if bool(template.get("version")) else "fail")
    add_check("metadata", "status is present", "pass" if bool(template.get("status")) else "fail")

    quality_rule_count = 0
    for obj in template.get("schema", []) or []:
        quality_rule_count += len(obj.get("quality", []) or [])
        for prop in obj.get("properties", []) or []:
            quality_rule_count += len(prop.get("quality", []) or [])
    add_check(
        "template_policy",
        "Template has no embedded quality rules",
        "pass" if quality_rule_count == 0 else "fail",
        severity="warning",
        detail=f"embedded_quality_rule_count={quality_rule_count}",
    )

    custom_props = template.get("customProperties") or []
    custom_map = {item.get("property"): item.get("value") for item in custom_props if isinstance(item, dict)}
    add_check("customProperties", "notebookValidationConfig present", "pass" if "notebookValidationConfig" in custom_map else "fail", severity="warning")
    add_check("customProperties", "ruleInjectionPolicy present", "pass" if "ruleInjectionPolicy" in custom_map else "fail", severity="warning")


TEMPLATE = load_template(TEMPLATE_PATH)
SCHEMA = load_schema(ODCS_SCHEMA_PATH)
if TEMPLATE and SCHEMA:
    validate_against_schema(TEMPLATE, SCHEMA)
if TEMPLATE:
    validate_template_conventions(TEMPLATE)

failed_errors = sum(1 for item in CHECKS if item["status"] == "fail" and item["severity"] == "error")
failed_warnings = sum(1 for item in CHECKS if item["status"] == "fail" and item["severity"] == "warning")
print(f"Template: {TEMPLATE_PATH}")
print(f"ODCS schema: {ODCS_SCHEMA_PATH}")
print(f"Checks: {len(CHECKS)}")
print(f"Failed errors: {failed_errors}")
print(f"Failed warnings: {failed_warnings}")

In [ ]:
output_path = Path(VALIDATION_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "validation_timestamp",
    "template_path",
    "odcs_schema_path",
    "section",
    "check_name",
    "status",
    "severity",
    "detail",
]
with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(CHECKS)

print(f"Validation checks exported to: {output_path}")

try:
    import pandas as pd
    result_df = pd.DataFrame(CHECKS)
    display(result_df)
except Exception:
    for row in CHECKS:
        print(row)

if FAIL_ON_ERROR and failed_errors > 0:
    raise RuntimeError(f"ODCS template validation failed with {failed_errors} error(s).")